# 02. A2 코넥스 선별 — 조회 결과 해석

**이 노트북이 하는 일:** 사용자가 2026-11-04 밤에 돌린 DART 조회 결과를 읽어서, **조회 전에 고정해 둔 규칙**(01번 노트북 2절)을 그대로 적용합니다. 규칙을 바꾸지 않았습니다.

| 항목 | 내용 |
|---|---|
| 입력 | `outputs/02_코넥스_선별/`의 `코넥스_선별대상_제조업.csv`, `코넥스_선별_수집로그.csv`, `코넥스_선별_결과_기업별.csv` |
| 출력 | 화면 요약만 |
| DART 호출 | **없음** |
| 먼저 알아야 할 것 | `01_선별대상_만들기.ipynb` (대상과 규칙) |

## 한 줄 결과
**코넥스 117곳 중 2019년 사업보고서가 조회된 기업은 11곳뿐이고, 그 11곳 중 2019년 분기 보고서 3개를 모두 가진 기업은 0곳입니다.** 사전기간 보고서 10개를 모두 가진 코넥스 기업도 0곳입니다. 같은 방식으로 조회한 코스닥 기업은 29곳 중 27곳(93%)이 분기 3개를 모두 가졌습니다.

다만 **분모가 11곳이라 사전에 정한 "20곳 미만이면 규칙을 적용하지 않고 사용자가 판단한다"는 예외 조항에 해당**합니다. 규칙이 자동으로 "제외"를 내린 것이 아니라, 사용자가 결정해야 합니다.

In [1]:
from pathlib import Path
import pandas as pd

BASE = Path.cwd().parents[1]
D = BASE / "outputs" / "02_코넥스_선별"
t = pd.read_csv(D / "코넥스_선별대상_제조업.csv", dtype=str, keep_default_na=False)
l = pd.read_csv(D / "코넥스_선별_수집로그.csv", dtype=str, keep_default_na=False)
r = pd.read_csv(D / "코넥스_선별_결과_기업별.csv", dtype=str, keep_default_na=False)
l["year"] = l["year"].astype(int)
print("조회 대상:", len(t), "곳 | 결과 행:", len(r), "| 로그 행:", len(l))
print("집단:", t["집단"].value_counts().to_dict())

조회 대상: 157 곳 | 결과 행: 157 | 로그 행: 2862
집단: {'코넥스': 117, '비교_코스닥': 40}


---
# 1. 조회가 제대로 끝났나 (먼저 검증)

결과를 해석하기 전에 **조회 자체가 완전했는지** 확인합니다. 빠진 기업이 있으면 해석이 틀어집니다.

In [2]:
NEED = [(2019, "Q1"), (2019, "H1"), (2019, "Q3"), (2019, "FY"), (2020, "Q1"), (2020, "H1"), (2020, "Q3"), (2020, "FY"), (2021, "Q1"), (2021, "H1")]
got = set(zip(l["corp_code"], l["year"], l["report"]))
incomplete = [c for c in t["corp_code"] if not all((c, y, rep) in got for y, rep in NEED)]
print("보고서 10개 모두에 대해 응답을 받은 기업:", len(t) - len(incomplete), "/", len(t), "(조회 미완료", len(incomplete), "곳)")
print("응답 상태:", l["status"].value_counts().to_dict(), " ← 000 정상, 013 조회된 데이터 없음")
print("응답 출처:", l["source"].value_counts().to_dict(), " ← api = 이번에 새로 요청, cache = 이미 저장된 것")
other = l[~l["status"].isin(["000", "013"])]
print("000·013 이외의 오류 응답:", len(other), "건")

보고서 10개 모두에 대해 응답을 받은 기업: 157 / 157 (조회 미완료 0 곳)
응답 상태: {'013': 2493, '000': 369}  ← 000 정상, 013 조회된 데이터 없음
응답 출처: {'api': 2782, 'cache': 80}  ← api = 이번에 새로 요청, cache = 이미 저장된 것
000·013 이외의 오류 응답: 0 건


### 해석
- **157곳 모두 조회가 끝났습니다.** 한도(020)나 오류로 멈춘 기업이 없습니다.
- 응답은 두 종류뿐입니다: **000(자료 있음)** 과 **013(조회된 데이터 없음)**. 013은 "DART API가 이 보고서를 응답으로 주지 않는다"는 뜻입니다.
- 새로 보낸 요청은 약 2,800건이고, 80건은 파일럿 때 이미 받아 둔 응답을 재사용했습니다.

---
# 2. 사전에 고정한 규칙을 그대로 적용

규칙(조회 전에 적어 둔 것, 임의 기준):
- **분모** = 2019년 사업보고서가 조회되는 코넥스 기업
- **분자** = 그중 2019년 1분기·반기·3분기 보고서가 모두 조회되는 기업
- **M = 분자 ÷ 분모**. M ≥ 50%이면 포함, 미만이면 제외
- **분모가 20곳 미만이면 규칙을 적용하지 않고 사용자가 판단**

In [3]:
tf = lambda s: s.str.lower().eq("true")
r["FY2019"] = tf(r["FY2019_있음"]); r["분기3개"] = tf(r["분기3개_2019_있음"]); r["10개모두"] = tf(r["10개_모두_있음"])
r["필요보고서수"] = r["필요보고서_개수(10개중)"].astype(int)

def metric(d):
    den = d[d["FY2019"]]; num = den[den["분기3개"]]
    return {"기업 수": len(d), "분모(2019 사업보고서 있음)": len(den), "분자(분기 3개 모두)": len(num),
            "M": f"{len(num)/len(den):.0%}" if len(den) else "-", "사전기간 10개 모두 있음": f"{int(d['10개모두'].sum())}곳 ({d['10개모두'].mean():.0%})"}

cx = r[r["집단"] == "코넥스"]
rows = {
    "코넥스 (전체)": metric(cx),
    "코넥스 (동명 모호 기업 제외)": metric(cx[cx["동명모호"] != "Y"]),
    "비교: 코스닥 40곳": metric(r[r["집단"] == "비교_코스닥"]),
}
pd.DataFrame(rows).T

,기업 수,분모(2019 사업보고서 있음),분자(분기 3개 모두),M,사전기간 10개 모두 있음
코넥스 (전체),117,11,0,0%,0곳 (0%)
코넥스 (동명 모호 기업 제외),110,11,0,0%,0곳 (0%)
비교: 코스닥 40곳,40,29,27,93%,27곳 (68%)


### 해석
| 집단 | 의미 |
|---|---|
| **코넥스** | 117곳 중 **2019년 사업보고서가 조회된 곳이 11곳**입니다(9%). 그 11곳 중 분기 3개를 모두 가진 곳은 **0곳**이므로 M은 0%입니다. 사전기간 10개 보고서를 모두 가진 곳도 **0곳**입니다 |
| 동명 모호 기업을 뺀 코넥스 | 7곳을 빼도 결과가 같습니다(분모 11, 분자 0). 같은 회사인지 모호한 기업 때문에 생긴 결과가 아닙니다 |
| **비교 코스닥** | 40곳 중 29곳이 2019년 사업보고서가 있고, 그중 27곳(93%)이 분기 3개를 모두 가졌습니다. 사전기간 10개 보고서를 모두 가진 곳은 27곳(68%)입니다 |

**규칙의 결론:** 분모가 **11곳으로 20곳 미만**이라서 사전에 정한 예외 조항에 해당합니다. 규칙은 "포함/제외"를 내리지 않고 **사용자의 판단**을 요청합니다.

그렇다고 결과가 애매한 것은 아닙니다. 같은 분모를 쓴 코스닥 비교 기업은 93%였고, 코넥스는 0곳입니다.

---
# 3. 코넥스 기업은 보고서가 얼마나 있나

In [4]:
print("[코넥스] 사전기간 필요 보고서 10개 중 조회된 개수별 기업 수")
print(cx["필요보고서수"].value_counts().sort_index().to_string())
print()
print("[비교 코스닥] 같은 분포")
print(r[r["집단"] == "비교_코스닥"]["필요보고서수"].value_counts().sort_index().to_string())

[코넥스] 사전기간 필요 보고서 10개 중 조회된 개수별 기업 수
필요보고서수
0    92
1     5
2     7
3     5
4     4
5     1
7     1
8     2

[비교 코스닥] 같은 분포
필요보고서수
0      9
3      2
7      1
8      1
10    27


### 해석
- **코넥스 117곳 중 92곳(79%)은 10개 보고서가 하나도 조회되지 않습니다.** 나머지도 1~8개이고, 10개를 모두 가진 곳은 없습니다.
- **비교 코스닥도 9곳은 0개**입니다. 이들은 사전기간 이후에 상장해서 2019~2020년 보고서가 없는 기업일 가능성이 있습니다(확인하지 않음). 그래서 분모를 "2019년 사업보고서가 있는 기업"으로 한정한 것입니다.
- 코스닥은 "있는 기업은 거의 다 갖췄고(10개 모두 27곳)", 코넥스는 "있는 기업도 거의 못 갖췄다"는 차이가 보입니다.

In [5]:
fy = cx[cx["FY2019"]]
rep = l[(l["status"] == "000") & (l["year"] == 2019) & (l["corp_code"].isin(fy["corp_code"]))]
print("2019년 사업보고서가 있는 코넥스 11곳에서, 2019년 보고서가 조회된 기업 수(보고서 종류별):")
print(rep.groupby("report")["corp_code"].nunique().to_dict())
print()
print("[출처별] 코넥스 집단")
g = cx.groupby("출처").apply(lambda d: pd.Series({"기업 수": len(d), "2019 사업보고서 있음": int(d["FY2019"].sum()), "분기 3개 모두": int(d["분기3개"].sum()), "필요 보고서 하나도 없음": int((d["필요보고서수"] == 0).sum())}), include_groups=False)
g

2019년 사업보고서가 있는 코넥스 11곳에서, 2019년 보고서가 조회된 기업 수(보고서 종류별):
{'FY': 11, 'Q3': 2}

[출처별] 코넥스 집단


,기업 수,2019 사업보고서 있음,분기 3개 모두,필요 보고서 하나도 없음
출처,,,,
KIND 시장 이전(코넥스),21,4,0,6
KIND 폐지(코넥스),34,1,0,31
현재 코넥스,62,6,0,55


### 해석
- **2019년 사업보고서가 있는 11곳에서도 2019년 1분기·반기 보고서는 한 곳도 조회되지 않았고, 3분기 보고서가 2곳에서만 조회되었습니다.** 즉 이 API에서 코넥스 기업은 사업보고서 위주로만 응답이 있고 분기 단위 자료는 거의 없습니다.
- **출처별로 보면** 현재 코넥스, KIND 폐지(코넥스), KIND 시장 이전(코넥스) 모두 비슷합니다. 시장 이전한 21곳 중 2019년 사업보고서가 있는 곳은 4곳뿐이고 분기 3개를 가진 곳은 0곳입니다. 즉 **코넥스에서 코스닥으로 옮긴 기업도 사전기간에 코넥스에 있던 동안의 분기 자료는 없습니다.**
- **KIND 폐지(코넥스) 34곳은 2019년 사업보고서가 있는 곳이 1곳뿐**입니다. 앞서 센 "제조업 코넥스 폐지 34건"은 대부분 이 분석에 쓸 자료가 없습니다.

---
# 4. 이 결과가 말하는 것과 말하지 않는 것

**말하는 것**
- 우리가 쓰려는 **DART API(fnlttSinglAcntAll)로는 코넥스 제조업 기업의 사전기간 분기 자료를 거의 얻을 수 없습니다.** 분기 단위 TTM ICR을 만들 수 없다는 뜻입니다.
- 이 결과는 **ICR 같은 결과값을 보지 않고** 얻었으므로, 이 결과로 코넥스를 제외해도 결과에 맞춘 표본 선택이 아닙니다.

**말하지 않는 것**
- 코넥스 기업이 **보고서를 제출하지 않았다**는 것은 아닙니다. "API가 응답을 주지 않는다"는 것뿐이고, 제출 의무(예: 코넥스는 분기 보고서가 면제되는지)는 이번에 확인하지 않았습니다.
- 코넥스 기업 전체의 비율이 아닙니다. 대상은 KIND 대조에서 이름이 확인된 기업과 현재 코넥스 기업 117곳입니다.
- 비교 코스닥은 **지금 살아 있는 기업**이라 코넥스 쪽보다 유리한 비교입니다.

---
# 5. 결정 선택지 (A2, 사용자 결정)

| 안 | 내용 | 영향 |
|---|---|---|
| **1. 코넥스 제외(제안)** | 코넥스 기업을 표본에서 제외한다. 이유는 "DART 분기 자료가 없어 TTM을 만들 수 없음"으로, 사전에 고정한 조회에서 나온 결과 | 이탈 후보도 줄어듦(인상 시작~분석 종료 폐지 39건 → 25건). 이 25건 중 실제로 DART 자료가 있는지는 본 수집에서 확인 |
| 2. 코넥스 포함 | 코넥스 기업을 대상에 넣어 두고 수집한다 | 분기 자료가 없어 **어차피 선별(사전기간 보고서 존재)에서 거의 다 탈락**합니다. 호출만 늘어남 |
| 3. 판단 보류 | 제조업 전체 선별 조회를 먼저 돌려 코넥스 64곳도 같이 확인한다 | 이미 조회한 코넥스 117곳과 같은 결과가 나올 가능성이 큼 |

**제안:** 1번입니다. 단, 이 결정은 **조회 결과를 본 뒤**에 내리는 것이므로 결정 기록에 "결과를 본 뒤: 예(단, ICR·노출도 관계는 보지 않음, 규칙의 20곳 미만 예외 조항에 따른 사용자 판단)"로 적어야 합니다.

### 한계
| 한계 | 이유 |
|---|---|
| 분모가 11곳으로 작음 | 규칙의 예외 조항(20곳 미만)에 해당. 통계적 비율이 아니라 사례 수준 |
| "API 응답이 없음 ≠ 제출 안 함" | 코넥스의 분기 보고서 제출 의무를 확인하지 않음 |
| 대상이 코넥스 전체가 아님 | 이름 대조로 확인된 기업 + 현재 코넥스 |
| 50%, 20곳은 임의 기준 | 조회 전에 적어 둔 값 |